# Setup: Config & Output Tables


## Imports

In [0]:
import pyspark.sql.functions as F

## Create the config (control) table
`hash_config` drives which tables get hashed, which columns form the primary key / composite key, and which hash algorithm to use.

In [0]:
config_table = "iics.casemix.hash_config"

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {config_table} (
    catalog         STRING,
    schema          STRING,
    table           STRING,
    primary_key     STRING,
    composite_keys  STRING,
    hash_algorithm  STRING,
    is_active       INT
)
USING DELTA
""")

### Seed sample config rows
Uses `MERGE` so re-running this cell is idempotent (no duplicate rows).

In [0]:
sample_config = [
    ("iics", "casemix", "healthcare_ingestion_test", "CASE_SURR_ID",
     "CASE_SURR_ID,PATIENT_SURR_ID,CASE_TYPE,HEALTHCARE_FACILITY,PATIENT_AC_NO,DIAG_CODE,DIAG_TYPE,MDC,DRG",
     "md5", 1),
    ("iics", "casemix", "healthcare_merge_test", "CASE_SURR_ID",
     "CASE_SURR_ID,PATIENT_SURR_ID,CASE_TYPE,HEALTHCARE_FACILITY,PATIENT_AC_NO,DIAG_CODE,DIAG_TYPE,MDC,DRG",
     "sha256", 1),
]

sample_config_df = spark.createDataFrame(
    sample_config,
    schema="catalog STRING, schema STRING, table STRING, primary_key STRING, "
           "composite_keys STRING, hash_algorithm STRING, is_active INT"
)

## Create the output (hashed values) table
One row is written per source row, per configured table. Columns ending in `___c` (e.g. `loaded_at___c`) are metadata columns and are always excluded from the columns that get hashed.

In [0]:
tgt_catalog = "iics" 
tgt_schema  = "casemix"
tgt_table   = "hashed_values"

In [0]:
output_table = f"{tgt_catalog}.{tgt_schema}.{tgt_table}"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {output_table} (
    catalog                     STRING,
    schema                      STRING,
    table                       STRING,
    primary_key                 STRING,
    primary_key_hash_value      STRING,
    composite_keys              STRING,
    composite_keys_hash_value   STRING,
    hash_value                  STRING,
    hash_algo                   STRING,
    loaded_at___c               TIMESTAMP
)
USING DELTA
""")

print(f"Output table ready: {output_table}")